# runtime

In [ ]:
open rust
open rust_operators
open sm'_operators

In [ ]:
///- --test static

open testing
open file_system_operators

## runtime

### split_args

In [ ]:
let split_args (args : string) : result (array_base string) _ =
    open parsing
    inl esc = [ '\\'; '`' ]
    inl quotes = [ '"' ]
    inl special = esc ++ quotes
    inl p_esc_pair (p : parser _) : parser string = fun t, i, s =>
        match p (t, i, s) with
        | Ok (_, i', s') => Ok (t |> slice i i', i', s')
        | Error e => Error e
    inl p_esc_char c = p_char c >>. any_char () |> p_esc_pair
    inl p_word = special |> none_of |>> fun c => fun () => c () |> sm'.obj_to_string
    inl p_plain = special ++ [ ' ' ] |> many1_none_of
    inl p_esc = esc |> listm.map p_esc_char |> choice
    inl p_chunk = special |> many1_none_of
    inl p_quoted = (p_chunk <|> p_esc) |> many_strings
    inl p_quoted_all = p_quoted |> between (p_char '"') (p_char '"')
    inl p_esc_root = p_esc >>% "" >>. p_word |> many |>> fun s => fun () => s () |> sm'.concat_list ""
    inl p_esc_quote_delim =
        esc
        |> listm.map fun c => p_char c >>. p_char '"' >>% '"'
        |> choice
    inl p_esc_char_noquote c = p_char c >>. none_of [ '"' ] |> p_esc_pair
    inl p_esc_noquote = esc |> listm.map p_esc_char_noquote |> choice
    inl p_quoted_esc = (p_chunk <|> p_esc_noquote) |> many_strings
    inl p_quoted_all_esc = p_quoted_esc |> between p_esc_quote_delim p_esc_quote_delim
    inl ensure_progress p : parser _ = fun input, i, s =>
        match p (input, i, s) with
        | Ok (r, i', s') when i' = i =>
            fail (fun _ => "runtime.split_args / segment zero-length success") i s
        | other => other
    inl p_segment = ensure_progress (p_quoted_all_esc <|> p_quoted_all <|> p_plain <|> p_esc_root)
    inl p_content = p_segment |> many1_strings
    inl p_args = spaces1 () |> sep_by p_content
    args
    |> run_parser p_args
    |> resultm.map (fst >> fun (args : list string) =>
        if is_rust () then
            inl n = (0i32, args) ||> listm.fold fun n _ => n + 1
            inl acc : array_base string = $'Rc::new(RefCell::new(Vec::with_capacity(!n as usize)))'
            inl _ : i32 = (0i32, args) ||> listm.fold fun n (x : string) =>
                $'!acc.borrow_mut().push(!x)' : ()
                n + 1
            acc
        else args |> listm'.box |> listm'.to_array'
    )
    |> resultm.map_error eval

In [1]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

Debug |> set_trace_level
[
    "a b c",
    ;[ "a"; "b"; "c" ]

    "e f \"g h\" i",
    ;[ "e"; "f"; "g h"; "i" ]

    "\"j k\" \"l\" \"m\"",
    ;[ "j k"; "l"; "m" ]

    "s -t \"u \`\"v\`\" w\"",
    ;[ "s"; "-t"; "u \`\"v\`\" w" ]

    "n -o \"p \\\"q\\\" r\"",
    ;[ "n"; "-o"; "p \\\"q\\\" r" ]

    "r -s \"t \\\"u\\\"\"",
    ;[ "r"; "-s"; "t \\\"u\\\"" ]

    $'"x -y \\\"$z -a \'(b=\\\\\\"c-id=)[a-fA-F0-9]{8}\', { \`$_[1] + \`$d++ }\\\""',
    ;[ "x"; "-y"; "$z -a '(b=\\\"c-id=)[a-fA-F0-9]{8}', { `$_[1] + `$d++ }" ]

    "e -f \"$g -h '(i=`\"j-id=)[a-fA-F0-9]{8}', { `$_[1] + `$k++ }\"",
    ;[ "e"; "-f"; "$g -h '(i=`\"j-id=)[a-fA-F0-9]{8}', { `$_[1] + `$k++ }" ]

    $'"--l \\\\\\"\'\'\' m \'\'\'\\\\\\" "',
    ;[ "--l"; "''' m '''" ]

    $'"n --o --p q --r \\\"s:/t u/v.w\\\" --x \\\"y:/z.a\\\" --b c.d \\\"\\\\e{f-g}\\\" h.i \\\"j (k)\\\""',
    ;[
        "n"; "--o"; "--p"; "q"; "--r"; "s:/t u/v.w"; "--x"; "y:/z.a"; "--b"; "c.d"; "\\e{f-g}"; "h.i"; "j (k)"
    ]

    $'"l \\\"m n:\\\\o.p\\\""',
    ;[ "l"; "m n:\\o.p" ]

    $'"--r s \\\\\\"\' t \'\\\\\\" u v w \\\\\\"x y\\\\\\""',
    ;[ "--r"; "s"; "' t '"; "u"; "v"; "w"; "x y" ]
]
|> _assert_fn split_args

.gleam (gleam)
13:01:40 1v _assert_fn / { input = a b c }
__assert_eq { expected = [a, b, c] }
13:01:40 1v _assert_fn / { input = e f "g h" i }
__assert_eq { expected = [e, f, g h, i] }
13:01:40 1v _assert_fn / { input = "j k" "l" "m" }
__assert_eq { expected = [j k, l, m] }
13:01:40 1v _assert_fn / { input = s -t "u `"v`" w" }
__assert_eq { expected = [s, -t, u `"v`" w] }
13:01:40 1v _assert_fn / { input = n -o "p \"q\" r" }
__assert_eq { expected = [n, -o, p \"q\" r] }
13:01:40 1v _assert_fn / { input = r -s "t \"u\"" }
__assert_eq { expected = [r, -s, t \"u\"] }
13:01:40 1v _assert_fn / { input = x -y "$z -a '(b=\"c-id=)[a-fA-F0-9]{8}', { `$_[1] + `$d++ }" }
__assert_eq { expected = [x, -y, $z -a '(b=\"c-id=)[a-fA-F0-9]{8}', { `$_[1] + `$d++ }] }
13:01:40 1v _assert_fn / { input = e -f "$g -h '(i=`"j-id=)[a-fA-F0-9]{8}', { `$_[1] + `$k++ }" }
__assert_eq { expected = [e, -f, $g -h '(i=`"j-id=)[a-fA-F0-9]{8}', { `$_[1] + `$k++ }] }
13:01:40 1v _assert_fn / { input = --l \"''' m '''\"

0

### split_command

In [ ]:
let split_command (command : string) : result (string * option string) _ =
    open parsing
    inl quotes = [ '"'; '\'' ]
    inl p_quoted_char = quotes |> listm.map p_char |> choice
    inl normalize = fun s => fun () => s () |> sm'.replace "\\" "/"
    inl p_quoted = quotes |> none_of |> many_chars |> between p_quoted_char p_quoted_char |>> normalize
    inl p_unquoted = quotes ++ [ ' ' ] |> none_of |> many1_chars |>> normalize
    inl p_path = p_quoted <|> p_unquoted <|> (eof () >>% "")
    inl p_args = spaces1 () >>. rest_of_line ()
    inl p_command = p_path .>>. (p_args |> opt)
    command
    |> run_parser p_command
    |> resultm.map (fst >> fun path, args => path (), args () |> optionm.map eval)
    |> resultm.map_error eval

In [2]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

Debug |> set_trace_level
[
    "",
    ("", None)

    "/a/b/c",
    ("/a/b/c", None)

    "d e.f",
    ("d", "e.f" |> Some)

    "..\\..\\g.h i.j k.l",
    ("../../g.h", "i.j k.l" |> Some)

    "m:\\n\\o.p \"q.r s.t\"",
    ("m:/n/o.p", "\"q.r s.t\"" |> Some)

    "\"..\\..\\u v\\w.x\" \"y z.a\" b.c",
    ("../../u v/w.x", "\"y z.a\" b.c" |> Some)

    "\"..\\..\\d e.f\" -g \\\\\"h i\\\\\"",
    ("../../d e.f", "-g \\\\\"h i\\\\\"" |> Some)

    "..\\..\\j k.l -m \\\\\"n o\\\\\"",
    ("../../j", "k.l -m \\\\\"n o\\\\\"" |> Some)
]
|> _assert_fn split_command

.gleam (gleam)
13:01:36 1v _assert_fn / { input =  }
__assert_eq { expected = (, None) }
13:01:36 1v _assert_fn / { input = /a/b/c }
__assert_eq { expected = (/a/b/c, None) }
13:01:36 1v _assert_fn / { input = d e.f }
__assert_eq { expected = (d, Some(e.f)) }
13:01:36 1v _assert_fn / { input = ..\..\g.h i.j k.l }
__assert_eq { expected = (../../g.h, Some(i.j k.l)) }
13:01:36 1v _assert_fn / { input = m:\n\o.p "q.r s.t" }
__assert_eq { expected = (m:/n/o.p, Some("q.r s.t")) }
13:01:36 1v _assert_fn / { input = "..\..\u v\w.x" "y z.a" b.c }
__assert_eq { expected = (../../u v/w.x, Some("y z.a" b.c)) }
13:01:36 1v _assert_fn / { input = "..\..\d e.f" -g \\"h i\\" }
__assert_eq { expected = (../../d e.f, Some(-g \\"h i\\")) }
13:01:36 1v _assert_fn / { input = ..\..\j k.l -m \\"n o\\" }
__assert_eq { expected = (../../j, Some(k.l -m \\"n o\\")) }
.fsx (fsharp)
__assert_eq { expected = (, None) }
__assert_eq { expected = (/a/b/c, None) }
__assert_eq { expected = (d, Some(e.f)) }
__assert_eq

0

### execution_line

In [ ]:
type execution_line =
    {
        process_id : int
        line : string
        error : bool
    }

## rust

### process_child

In [ ]:
nominal process_child =
    `(
        rust_type
            `($'std_process_Child')
            `($'std::process::Child')
            "type std_process_Child = class end"
    )

### process_child_stdin

In [ ]:
nominal process_child_stdin =
    `(
        rust_type
            `($'std_process_ChildStdin')
            `($'std::process::ChildStdin')
            "type std_process_ChildStdin = class end"
    )

inl process_child_stdin
    (child : rust.ref (rust.mut' process_child))
    : rust.ref (rust.mut' (optionm'.option' process_child_stdin))
    =
    !\\(child, $'"&mut $0.stdin"')

## runtime

### execution_options

In [ ]:
type execution_options =
    {
        command : string
        cancellation_token : optionm'.option' threading.cancellation_token
        environment_variables : array_base (string * string)
        on_line : optionm'.option' (execution_line -> async.async ())
        stdin : optionm'.option' (threading.arc (threading.mutex process_child_stdin) -> ())
        trace : bool
        working_directory : optionm'.option' string
        stderr : bool
    }

inl execution_options (fn : execution_options -> execution_options) : execution_options =
    {
        command = ""
        cancellation_token = None |> optionm'.box
        environment_variables = ;[]
        on_line = None |> optionm'.box
        stdin = None |> optionm'.box
        trace = true
        working_directory = None |> optionm'.box
        stderr = true
    }
    |> fn

## rust

### process_child_stderr

In [ ]:
nominal process_child_stderr =
    `(
        rust_type
            `($'std_process_ChildStderr')
            `($'std::process::ChildStderr')
            "type std_process_ChildStderr = class end"
    )

inl process_child_stderr
    (child : rust.ref (rust.mut' process_child))
    : rust.ref (rust.mut' (optionm'.option' process_child_stderr))
    =
    !\\(child, $'"&mut $0.stderr"')

### process_child_stdout

In [ ]:
nominal process_child_stdout =
    `(
        rust_type
            `($'std_process_ChildStdout')
            `($'std::process::ChildStdout')
            "type std_process_ChildStdout = class end"
    )

inl process_child_stdout
    (child : rust.ref (rust.mut' process_child))
    : rust.ref (rust.mut' (optionm'.option' process_child_stdout))
    =
    !\\(child, $'"&mut $0.stdout"')

### process_command

In [ ]:
nominal process_command =
    `(
        rust_type
            `($'std_process_Command')
            `($'std::process::Command')
            "type std_process_Command = class end"
    )

### process_stdio

In [ ]:
nominal process_stdio =
    `(
        rust_type
            `($'std_process_Stdio')
            `($'std::process::Stdio')
            "type std_process_Stdio = class end"
    )

### process_output

In [ ]:
nominal process_output =
    `(
        rust_type
            `($'std_process_Output')
            `($'std::process::Output')
            "type std_process_Output = class end"
    )

### process_exit_status

In [ ]:
nominal process_exit_status =
    `(
        rust_type
            `($'std_process_ExitStatus')
            `($'std::process::ExitStatus')
            "type std_process_ExitStatus = class end"
    )

### process_output_status

In [ ]:
inl process_output_status (output : process_output) : process_exit_status =
    !\\(output, $'"$0.status"')

### process_output_stdout

In [ ]:
inl process_output_stdout (output : process_output) : am'.vec u8 =
    !\\(output, $'"$0.stdout"')

### process_output_stderr

In [ ]:
inl process_output_stderr (output : process_output) : am'.vec u8 =
    !\\(output, $'"$0.stderr"')

### process_exit_status_code

In [ ]:
inl process_exit_status_code (status : process_exit_status) : optionm'.option' i32 =
    !\\(status, $'"$0.code()"')

### stdin_write_all

In [ ]:
inl stdin_write_all (stdin : threading.mutex_guard process_child_stdin) (text : string) : () =
    backend_switch {
        Fsharp = fun () =>
            inl stream = text |> sm'.as_bytes
            inl stdin = join stdin
            (!\($'"true; let mut !stdin = !stdin"') : bool) |> ignore
            (!\\(stdin, $'"true; std::io::Write::write_all(&mut *$0, !stream).unwrap()"') : bool) |> ignore
        Rust = fun () =>
            inl stream = text |> sm'.as_bytes
            (!\\((stdin, stream), $'"true; std::io::Write::write_all(&mut *$0, $1).unwrap()"') : bool) |> ignore
    }

### stdin_flush

In [ ]:
inl stdin_flush (stdin : threading.mutex_guard process_child_stdin) : () =
    backend_switch {
        Fsharp = fun () =>
            inl stdin = join stdin
            (!\($'"true; let mut !stdin = !stdin"') : bool) |> ignore
            (!\\(stdin, $'"true; std::io::Write::flush(&mut *$0).unwrap()"') : bool) |> ignore
        Rust = fun () =>
            (!\\(stdin, $'"true; std::io::Write::flush(&mut *$0).unwrap()"') : bool) |> ignore
    }

### new_process_command

In [ ]:
inl new_process_command (file_name : string) : process_command =
    !\\(file_name, $'"std::process::Command::new(&*$0)"')

### process_stdio_piped

In [ ]:
inl process_stdio_piped () : process_stdio =
    !\($'"std::process::Stdio::piped()"')

### process_command_args

In [ ]:
inl process_command_args (args : am'.vec sm'.std_string) (c : process_command) : process_command =
    (!\($'"true; let mut !c = !c"') : bool) |> ignore
    (!\\(args, $'"true; std::process::Command::args(&mut !c, &*$0)"') : bool) |> ignore
    c |> rust.emit

### process_command_stdout

In [ ]:
inl process_command_stdout (stdio : process_stdio) (c : process_command) : process_command =
    (!\($'"true; let mut !c = !c"') : bool) |> ignore
    (!\($'"true; std::process::Command::stdout(&mut !c, std::process::Stdio::piped())"') : bool) |> ignore
    c |> rust.emit

### process_command_stderr

In [ ]:
inl process_command_stderr (stdio : process_stdio) (c : process_command) : process_command =
    (!\($'"true; let mut !c = !c"') : bool) |> ignore
    (!\($'"true; std::process::Command::stderr(&mut !c, std::process::Stdio::piped())"') : bool) |> ignore
    c |> rust.emit

### process_command_stdin

In [ ]:
inl process_command_stdin (stdio : process_stdio) (c : process_command) : process_command =
    (!\($'"true; let mut !c = !c"') : bool) |> ignore
    (!\($'"true; std::process::Command::stdin(&mut !c, std::process::Stdio::piped())"') : bool) |> ignore
    c |> rust.emit

### process_command_current_dir

In [ ]:
inl process_command_current_dir (dir : string) (c : process_command) : process_command =
    (!\($'"true; let mut !c = !c"') : bool) |> ignore
    (!\\(dir, $'"true; std::process::Command::current_dir(&mut !c, &*$0)"') : bool) |> ignore
    !\($'$"!c"')

### process_command_env

In [ ]:
inl process_command_env (key : string) (value : string) (c : process_command) : process_command =
    (!\($'"true; let mut !c = !c"') : bool) |> ignore
    (!\\((key, value), $'"true; std::process::Command::env(&mut !c, &*$0, &*$1)"') : bool) |> ignore
    c |> rust.emit

### process_command_spawn

In [ ]:
inl process_command_spawn
    (c : process_command)
    : resultm.result' process_child stream.io_error
    =
    (!\($'"true; let mut !c = !c"') : bool) |> ignore
    !\($'"std::process::Command::spawn(&mut !c)"')

### child_wait_with_output

In [ ]:
inl child_wait_with_output
    (child : process_child)
    : resultm.result' process_output stream.io_error
    =
    !\\(child, $'"$0.wait_with_output()"')

### stdio_line

In [ ]:
inl stdio_line
    (stdio : result () ())
    (trace' : bool)
    (channel_sender : threading.arc (threading.mutex (threading.channel_sender sm'.std_string)))
    (line : resultm.result' sm'.std_string stream.io_error)
    : resultm.result' () sm'.std_string
    =
    inl highlight text =
        $'$"\\u001b[4;7m{!text}\\u001b[0m"'
    inl line =
        match
            line
            |> resultm.map_error'' sm'.format'
            |> resultm.unbox'
        with
        | Ok line =>
            inl line =
                line
                |> sm'.from_std_string
                |> sm'.encoding_encode' (sm'.encoding_utf8' ())
                |> rust.cow_as_ref
                |> sm'.str_from_utf8
                |> resultm.unwrap'
                |> sm'.ref_to_std_string
            inl line_log = line |> sm'.from_std_string
            inl text =
                match stdio with
                | Ok () => $'$"> {!line_log}"'
                | Error () => $'$"\! {!line_log}"'
            if trace'
            then trace Verbose (fun () => text) id
            else text |> console.write_line
            match stdio with
            | Ok () => line
            | Error () => line |> highlight |> sm'.to_std_string
        | Error e =>
            trace Critical
                fun () => "runtime.stdio_line"
                fun () => { trace' e }
            e |> highlight |> sm'.to_std_string
    channel_sender
    |> threading.arc_mutex_lock
    |> resultm.unwrap'
    |> threading.mutex_guard_ref
    |> threading.channel_send line
    |> resultm.map_error'' sm'.format'

### stdio_line_rust

In [ ]:
inl stdio_line_rust
    (stdio : result () ())
    (trace' : bool)
    (line : resultm.result' sm'.std_string sm'.std_string)
    : sm'.std_string
    =
    inl highlight text =
        $'$"\\u001b[4;7m{!text}\\u001b[0m"'
    match line |> resultm.unbox' with
    | Ok line =>
        inl line_log = line |> rust.clone |> sm'.from_std_string
        inl text =
            match stdio with
            | Ok () => $'$"> {!line_log}"'
            | Error () => $'$"\! {!line_log}"'
        if trace'
        then trace Verbose (fun () => text) id
        else text |> console.write_line
        match stdio with
        | Ok () => line
        | Error () => line |> highlight |> sm'.to_std_string
    | Error e =>
        trace Critical
            fun () => "runtime.stdio_line"
            fun () => { trace' e }
        e |> highlight |> sm'.to_std_string

### command

In [ ]:
nominal command =
    `(
        rust_type
            `($'clap_Command')
            `($'clap::Command')
            "type clap_Command = class end"
    )

### new_command

In [ ]:
inl new_command (s : rust.static_ref sm'.str) : command =
    !\\(s, $'"clap::Command::new($0)"')

In [3]:
///- --test
///> rust -d clap

###"command"
|> new_command
|> sm'.format_pretty
|> _assert sm'.contains "\"command\""

__assert { expected = Command {
    name: "command",
    long_flag: None,
    short_flag: None,
    display_name: None,
    bin_name: None,
    author: None,
    version: None,
    long_version: None,
    about: None,
    long_about: None,
    before_help: None,
    before_long_help: None,
    after_help: None,
    after_long_help: None,
    aliases: [],
    short_flag_aliases: [],
    long_flag_aliases: [],
    usage_str: None,
    usage_name: None,
    help_str: None,
    disp_ord: None,
    template: None,
    settings: AppFlags(
        0,
    ),
    g_settings: AppFlags(
        0,
    ),
    args: MKeyMap {
        args: [],
        keys: [],
    },
    subcommands: [],
    groups: [],
    current_help_heading: None,
    current_disp_ord: Some(
        0,
    ),
    subcommand_value_name: None,
    subcommand_heading: None,
    external_value_parser: None,
    long_help_exists: false,
    deferred: None,
    app_ext: Extensions {
        extensions: FlatMap {
            keys: []

### arg

In [ ]:
nominal arg =
    `(
        rust_type
            `($'clap_Arg')
            `($'clap::Arg')
            "type clap_Arg = class end"
    )

### new_arg

In [ ]:
inl new_arg (s : rust.static_ref sm'.str) : arg =
    !\\(s, $'"clap::Arg::new($0)"')

In [4]:
///- --test
///> rust -d clap

###"arg"
|> new_arg
|> sm'.format_pretty
|> _assert sm'.contains "\"arg\""

__assert { expected = Arg {
    id: "arg",
    help: None,
    long_help: None,
    action: None,
    value_parser: None,
    conflicts: [],
    settings: ArgFlags(
        0,
    ),
    overrides: [],
    groups: [],
    requires: [],
    r_ifs: [],
    r_unless: [],
    short: None,
    long: None,
    aliases: [],
    short_aliases: [],
    disp_ord: None,
    val_names: [],
    num_vals: None,
    val_delim: None,
    default_vals: [],
    default_vals_ifs: [],
    terminator: None,
    index: None,
    help_heading: None,
    default_missing_vals: [],
    ext: Extensions {
        extensions: FlatMap {
            keys: [],
            values: [],
        },
    },
} }

### command_arg

In [ ]:
inl command_arg (arg : arg) (command : command) : command =
    !\\((command, arg), $'"clap::Command::arg($0, $1)"')

### arg_required

In [ ]:
inl arg_required (value : bool) (arg : arg) : arg =
    !\\((arg, value), $'"$0.required($1)"')

### arg_require_equals

In [ ]:
inl arg_require_equals (value : bool) (arg : arg) : arg =
    !\\((arg, value), $'"$0.require_equals($1)"')

### arg_default_value

In [ ]:
inl arg_default_value (value : string) (arg : arg) : arg =
    inl value = #value
    !\\((arg, value), $'"$0.default_value(&*Box::leak(String::from($1).into_boxed_str()))"')

### arg_default_missing_value

In [ ]:
inl arg_default_missing_value (value : string) (arg : arg) : arg =
    inl value = #value
    backend_switch {
        Rust = fun () => !\\((arg, value), $'"$0.default_missing_value(&*Box::leak(String::from($1).into_boxed_str()))"') : arg
    }

### arg_overrides_with

In [ ]:
inl arg_overrides_with (value : string) (arg : arg) : arg =
    inl value = #value
    !\\((arg, value), $'"$0.overrides_with($1)"')

### arg_short

In [ ]:
inl arg_short (value : char) (arg : arg) : arg =
    !\\((arg, value), $'"$0.short($1 as char)"')

### arg_long

In [ ]:
inl arg_long (value : rust.static_ref sm'.str) (arg : arg) : arg =
    !\\((arg, value), $'"$0.long($1)"')

### arg_value_names

In [ ]:
inl arg_value_names (values : array_base (rust.static_ref sm'.str)) (arg : arg) : arg =
    inl values = values |> am'.to_vec
    !\\((arg, values), $'"$0.value_names($1)"')

### arg_num_args

In [ ]:
inl arg_num_args (value : i32) (arg : arg) : arg =
    !\\((arg, value), $'"$0.num_args($1)"')

### value_range

In [ ]:
nominal value_range =
    `(
        rust_type
            `($'clap_builder_ValueRange')
            `($'clap::builder::ValueRange')
            "type clap_builder_ValueRange = class end"
    )

### new_value_range

In [ ]:
inl new_value_range forall t. inclusive (start : _ t) (end : _ t) : value_range =
    inl len () =
        0i32 |> convert.convert
    inl start, end =
        open am'
        match start, end with
        | Start start, End fn =>
            start, len |> fn
        | End start_fn, End end_fn =>
            start_fn len, end_fn len
    inl inclusive =
        if inclusive
        then "="
        else ""
    match start, end with
    | start, end when end =. len () => !\\(start, $'"clap::builder::ValueRange::new($0..)"')
    | start, end => !\\((start, end), $'"clap::builder::ValueRange::new($0.." + !inclusive + "$1)"')

### arg_num_args_range

In [ ]:
inl arg_num_args_range (value : value_range) (arg : arg) : arg =
    !\\((arg, value), $'"$0.num_args($1)"')

### arg_value_name

In [ ]:
inl arg_value_name (value : string) (arg : arg) : arg =
    inl value = value |> sm'.as_str
    !\\((arg, value), $'"$0.value_name($1)"')

### value_parser

In [ ]:
nominal value_parser =
    `(
        rust_type
            `($'clap_builder_ValueParser')
            `($'clap::builder::ValueParser')
            "type clap_builder_ValueParser = class end"
    )

### possible_value

In [ ]:
nominal possible_value =
    `(
        rust_type
            `($'clap_builder_PossibleValue')
            `($'clap::builder::PossibleValue')
            "type clap_builder_PossibleValue = class end"
    )

### new_possible_value

In [ ]:
inl new_possible_value forall t. (x : t) : possible_value =
    !\\(x, $'"clap::builder::PossibleValue::new(&**$0)"')

### value_parser_path_buf

In [ ]:
inl value_parser_path_buf () : value_parser =
    !\($'"clap::value_parser\!(std::path::PathBuf)"')

### value_parser_expr

In [ ]:
inl value_parser_expr (expr : string) : value_parser =
    !\($'"clap::value_parser\!(" + !expr + ").into()"')

### arg_value_parser

In [ ]:
inl arg_value_parser (values : value_parser) (arg : arg) : arg =
    !\\((arg, values), $'"$0.value_parser($1)"')

### arg_action

In [ ]:
nominal arg_action' =
    `(
        rust_type
            `($'clap_ArgAction')
            `($'clap::ArgAction')
            "type clap_ArgAction = class end"
    )

union arg_action =
    | Set
    | Append
    | SetTrue
    | SetFalse
    | Count
    | Help
    | HelpShort
    | HelpLong
    | Version

inl arg_action = function
    | Set => !\($'"clap::ArgAction::Set"') : arg_action'
    | Append => !\($'"clap::ArgAction::Append"') : arg_action'
    | SetTrue => !\($'"clap::ArgAction::SetTrue"') : arg_action'
    | SetFalse => !\($'"clap::ArgAction::SetFalse"') : arg_action'
    | Count => !\($'"clap::ArgAction::Count"') : arg_action'
    | Help => !\($'"clap::ArgAction::Help"') : arg_action'
    | HelpShort => !\($'"clap::ArgAction::HelpShort"') : arg_action'
    | HelpLong => !\($'"clap::ArgAction::HelpLong"') : arg_action'
    | Version => !\($'"clap::ArgAction::Version"') : arg_action'

inl arg_action (value : arg_action) (arg : arg) : arg =
    inl value = value |> arg_action
    !\\((arg, value), $'"$0.action($1)"')

### arg_index

In [ ]:
inl arg_index (value : i32) (arg : arg) : arg =
    inl index : convert.unativeint =
        if is_rust ()
        then $'(!value as usize)'
        else $'!value |> `(convert.unativeint) '
    !\\((arg, index), $'"$0.index($1)"')

### arg_matches

In [ ]:
nominal arg_matches =
    `(
        rust_type
            `($'clap_ArgMatches')
            `($'clap::ArgMatches')
            "type clap_ArgMatches = class end"
    )

### command_get_matches

In [ ]:
inl command_get_matches (command : command) : arg_matches =
    !\\(command, $'"clap::Command::get_matches($0)"')

### command_get_matches_from

In [ ]:
inl command_get_matches_from (args : array_base string) (command : command) : arg_matches =
    inl args = args |> am'.to_vec |> am'.vec_map sm'.to_std_string
    !\\(command, $'"clap::Command::get_matches_from($0, !args)"')

### command_args_override_self

In [ ]:
inl command_args_override_self (yes : bool) (command : command) : command =
    !\\(command, $'"clap::Command::args_override_self($0, !yes)"')

### command_init_arg

In [ ]:
inl command_init_arg (long, short) fn command =
    command
    |> command_arg (
        ###long
        |> new_arg
        |> arg_short short
        |> arg_long ###long
        |> fn
    )

### matches_get_one

In [ ]:
inl matches_get_one forall t. (x : string) (matches : arg_matches) : optionm'.option' t =
    inl x = join x
    inl x = x |> sm'.as_str
    !\\((matches, x), $'"clap::ArgMatches::get_one(&$0, $1).cloned()"')

### matches_get_flag

In [ ]:
inl matches_get_flag (x : string) (matches : arg_matches) : bool =
    inl x = join x
    inl x = x |> sm'.as_str
    !\\((matches, x), $'"clap::ArgMatches::get_flag(&$0, $1)"')

### matches_get_many

In [ ]:
inl matches_get_many forall t. (x : string) (matches : arg_matches) : optionm'.option' (am'.vec t) =
    inl x = join x
    inl x = x |> sm'.as_str
    !\\((matches, x), $'"clap::ArgMatches::get_many(&$0, $1).map(|x| x.cloned().into_iter().collect())"')

### matches_get_occurrences

In [ ]:
inl matches_get_occurrences (x : string) (matches : arg_matches) : optionm'.option' (array_base sm'.std_string) =
    inl x = join x
    inl x = x |> sm'.as_str
    !\($'"clap::ArgMatches::get_occurrences(&!matches, !x).cloned()"')

### matches_subcommand

In [ ]:
inl matches_subcommand (matches : arg_matches) : optionm'.option' (sm'.std_string * arg_matches) =
    !\\((matches, sm'.ref_to_std_string), $'"clap::ArgMatches::subcommand(Box::leak(Box::new($0))).map(|(a, b)| ($1(a), b.clone()))"')

### matches_values_of

In [ ]:
inl matches_values_of (x : string) (matches : arg_matches) : array_base sm'.std_string =
    !\\((matches, x), $'"clap::ArgMatches::values_of($0, &*$1)"')

### command_subcommand_required

In [ ]:
inl command_subcommand_required (value : bool) (command : command) : command =
    !\\(command, $'"clap::Command::subcommand_required($0, !value)"')

### command_subcommand

In [ ]:
inl command_subcommand (subcommand : command) (command : command) : command =
    !\\(command, $'"clap::Command::subcommand($0, !subcommand)"')

### value_parser_possible_values

In [ ]:
inl value_parser_possible_values (values : array_base string) : value_parser =
    inl values =
        values
        |> am'.to_vec
        |> am'.vec_map (sm'.to_std_string >> rust.new_box >> rust.box_leak >> new_possible_value)
    !\\(values, $'"Into::<clap::builder::ValueParser>::into(clap::builder::PossibleValuesParser::new($0))"')

### arg_union

In [ ]:
inl arg_union forall union_type. (fn : union_type -> ()) (arg : arg) : arg =
    arg
    |> arg_value_parser (
        real reflection.get_union_fields_untag `union_type ()
        |> fun x => x : _ (string * union_type)
        |> listm.map (fst >> sm'.to_lower)
        |> listm'.box
        |> listm'.to_array'
        |> value_parser_possible_values
    )

In [5]:
///- --test
///> rust -d clap

###"command"
|> new_command
|> command_init_arg ("trace-level", 't') (
    real arg_union `trace_level ignore
)
|> command_get_matches_from ;[ "_"; "--trace-level"; "critical" ]
|> matches_get_one "trace-level"
|> optionm'.unwrap
|> sm'.from_std_string
|> reflection.union_try_pick
|> optionm.value
|> _assert_eq Critical

__assert_eq { expected = Critical }

### command_debug_assert

In [ ]:
inl command_debug_assert (command : command) : () =
    !\\(command, $'"clap::Command::debug_assert($0)"')

## fsharp

### process

In [ ]:
nominal process = $'System.Diagnostics.Process'

### process_start_info

In [ ]:
nominal process_start_info = $'System.Diagnostics.ProcessStartInfo'

### data_received_event_args

In [ ]:
nominal data_received_event_args = $'System.Diagnostics.DataReceivedEventArgs'

### new_process

In [ ]:
inl new_process (process_start_info : process_start_info) : process =
    $'new `process (StartInfo = !process_start_info)'

### process_start

In [ ]:
inl process_start (process : process) : bool =
    $'!process.Start' ()

### process_exit_code

In [ ]:
inl process_exit_code (process : process) : i32 =
    run_target function
        | Fsharp => fun () => $'!process.ExitCode'
        | _ => fun () => null ()

### process_id

In [ ]:
let process_id (process : process) : i32 =
    run_target function
        | Fsharp => fun () => process |> $'_.Id'
        | _ => fun () => null ()

### process_has_exited

In [ ]:
let process_has_exited (process : process) : bool =
    run_target function
        | Fsharp => fun () => process |> $'_.HasExited'
        | _ => fun () => null ()

### process_kill

In [ ]:
let process_kill (process : process) : () =
    run_target function
        | Fsharp => fun () => process |> $'_.Kill()'
        | _ => fun () => ()

### process_begin_error_read_line

In [ ]:
inl process_begin_error_read_line (process : process) : () =
    process |> $'_.BeginErrorReadLine()'

### process_begin_output_read_line

In [ ]:
inl process_begin_output_read_line (process : process) : () =
    process |> $'_.BeginOutputReadLine()'

### process_add_output_data_received

In [ ]:
inl process_add_output_data_received fn (process : process) : () =
    $'!process.OutputDataReceived.Add !fn '

### process_add_error_data_received

In [ ]:
inl process_add_error_data_received fn (process : process) : () =
    $'!process.ErrorDataReceived.Add !fn '

### process_wait_for_exit_async

In [ ]:
inl process_wait_for_exit_async (ct : threading.cancellation_token) (process : process) : async.task () =
    run_target function
        | Fsharp => fun () => $'!process.WaitForExitAsync !ct '
        | _ => fun () => null ()

### event_data

In [ ]:
let event_data (e : data_received_event_args) : string =
    run_target function
        | Fsharp => fun () => e |> $'_.Data'
        | _ => fun () => null ()

### current_process_kill

In [ ]:
let current_process_kill () =
    run_target function
        | Fsharp => fun () =>
            inl fn () =
                run_target function
                    | Fsharp => fun () =>
                        trace Warning (fun () => "runtime.current_process_kill / exiting... 3") id
                        $'System.Threading.Thread.Sleep 500'
                        trace Warning (fun () => "runtime.current_process_kill / exiting... 2") id
                        $'System.Console.Out.Flush ()'
                        $'System.Threading.Thread.Sleep 1000'
                        trace Warning (fun () => "runtime.current_process_kill / exiting... 1") id
                        $'System.Console.Error.Flush ()'
                        $'System.Environment.Exit 1' : ()
                    | _ => fun () => ()
            inl thread : threading.thread = $'new System.Threading.Thread (!fn)'
            thread |> $'_.Start()' : ()
        | _ => fun () => ()

### gc_collect

In [ ]:
inl gc_collect () =
    run_target function
        | Fsharp _ => fun () => $'System.GC.Collect' () : ()
        | _ => fun () => ()

## runtime

### execute_with_options_fsharp

In [ ]:
inl execute_with_options_fsharp (options : execution_options) : i32 * string =
    inl command = join options.command
    inl file_name, arguments = command |> split_command |> resultm.get
    inl arguments =
        arguments
        |> optionm'.default_value ""
        |> split_args
        |> resultm.get
        |> am'.to_vec
        |> am'.vec_map sm'.to_std_string
    trace Debug
        fun () => "runtime.execute_with_options"
        fun () => { file_name arguments = #?*arguments; options }
    fun () =>
        fun () =>
            file_name
            |> new_process_command
            |> process_command_args arguments
            |> process_command_stdout (process_stdio_piped ())
            |> process_command_stderr (process_stdio_piped ())
            |> process_command_stdin (process_stdio_piped ())
            |> fun command =>
                match options.working_directory |> optionm'.unbox with
                | Some working_directory => command |> process_command_current_dir working_directory
                | None => !\($'$"!command"') |> rust.emit
            |> fun command =>
                match options.environment_variables with
                | ;[] => command
                | vars =>
                    (command, vars |> am'.to_vec)
                    ||> am'.vec_fold' fun command (key, value) =>
                        command |> process_command_env key value
            |> process_command_spawn
            |> resultm.map_error'' sm'.format'
            |> resultm.map' (rust.emit >> optionm'.some' >> rust.emit >> threading.new_arc_mutex)
            |> resultm.unbox'
            |> function
                | Ok child =>
                    inl stdout =
                        fun () =>
                            child
                            |> threading.arc_mutex_lock
                            |> resultm.unwrap'
                            |> threading.mutex_guard_ref_mut''
                            |> optionm'.as_mut
                            |> optionm'.unwrap
                            |> process_child_stdout
                            |> optionm'.take_ref_mut
                            |> optionm'.unwrap
                        |> rust.capture
                    inl stderr =
                        fun () =>
                            child
                            |> threading.arc_mutex_lock
                            |> resultm.unwrap'
                            |> threading.mutex_guard_ref_mut''
                            |> optionm'.as_mut
                            |> optionm'.unwrap
                            |> process_child_stderr
                            |> optionm'.take_ref_mut
                            |> optionm'.unwrap
                        |> rust.capture
                    inl stdin =
                        fun () =>
                            child
                            |> threading.arc_mutex_lock
                            |> resultm.unwrap'
                            |> threading.mutex_guard_ref_mut''
                            |> optionm'.as_mut
                            |> optionm'.unwrap
                            |> process_child_stdin
                            |> optionm'.take_ref_mut
                            |> optionm'.unwrap
                            |> optionm'.some'
                            |> rust.emit
                            |> threading.new_arc_mutex
                        |> rust.capture
                    inl channel_sender, channel_receiver = threading.new_channel ()
                    inl channel_sender'' = channel_sender |> rust.emit |> threading.new_arc_mutex
                    inl channel_sender' = channel_sender |> rust.emit |> threading.new_arc_mutex
                    inl channel_receiver' = channel_receiver |> rust.emit |> threading.new_arc_mutex
                    inl stdout_handle =
                        fun () =>
                            stdout
                            |> stream.decode_reader_bytes_build
                            |> stream.new_buf_reader
                            |> stream.buf_read_lines
                            |> iter.try_for_each fun lines =>
                                inl channel_sender'' = channel_sender'' |> rust.clone
                                lines
                                |> stdio_line (Ok ()) options.trace channel_sender''
                                |> resultm.to_try
                        |> threading.spawn (1, 0) 1
                    inl stderr_handle =
                        fun () =>
                            stderr
                            |> stream.decode_reader_bytes_build
                            |> stream.new_buf_reader
                            |> stream.buf_read_lines
                            |> iter.try_for_each fun lines =>
                                if options.stderr |> not
                                then Ok () |> resultm.box
                                else
                                    inl channel_sender' = channel_sender' |> rust.clone
                                    lines
                                    |> stdio_line (Error ()) options.trace channel_sender'
                                |> resultm.to_try
                        |> threading.spawn (1, 0) 1
                    match options.stdin |> optionm'.unbox with
                    | Some stdin' =>
                        stdin
                        |> threading.arc_mutex_lock
                        |> resultm.unwrap'
                        |> threading.mutex_guard_ref_mut''
                        |> optionm'.take_ref_mut
                        |> optionm'.map' threading.new_arc_mutex
                        |> optionm'.unbox
                        |> function
                            | Some stdin =>
                                stdin |> stdin'
                                stdin
                                |> threading.arc_mutex_lock'
                                |> resultm.unwrap'
                                |> stdin_flush
                            | None => ()
                    | None => ()
                    inl output =
                        child
                        |> threading.arc_mutex_lock'
                        |> resultm.unwrap'
                        |> threading.mutex_guard_ref_mut''
                        |> optionm'.take_ref_mut
                        |> optionm'.unwrap
                        |> child_wait_with_output
                        |> resultm.map_error'' sm'.format'
                    [ stdout_handle; stderr_handle ]
                    |> am'.new_vec
                    |> am'.vec_for_each' (threading.join' >> resultm.unwrap' >> resultm.unwrap')
                    match output |> resultm.unbox with
                    | Ok output =>
                        inl exit_code =
                            output
                            |> process_output_status
                            |> process_exit_status_code
                            |> optionm'.unbox
                        match exit_code with
                        | Some exit_code =>
                            inl stdout =
                                output
                                |> process_output_stdout
                                |> sm'.string_from_utf8
                                |> resultm.unwrap'
                            inl stderr =
                                output
                                |> process_output_stderr
                                |> sm'.string_from_utf8
                                |> resultm.unwrap'
                            inl output =
                                [ stdout; stderr ]
                                |> listm.map (sm'.from_std_string >> sm'.trim)
                                |> listm'.filter (sm'.is_empty >> not)
                                |> sm'.concat_list "\n"
                                |> sm'.to_std_string
                                |> Some
                            exit_code, output, Some channel_receiver'
                        | None =>
                            -1,
                            ("runtime.execute_with_options / exit_code=None"
                                |> sm'.to_std_string
                                |> Some),
                            Some channel_receiver'
                    | Error error =>
                        trace Critical
                            fun () => "runtime.execute_with_options / output error"
                            fun () => { error file_name arguments = #?*arguments; options }
                        -2i32, error |> Some, None
                | Error error =>
                    trace Critical
                        fun () => "runtime.execute_with_options / child error"
                        fun () => { error file_name arguments = #?*arguments; options }
                    -1i32, error |> Some, None
            |> function
                | exit_code, std_trace, channel_receiver =>
                    inl std_trace =
                        channel_receiver
                        |> optionm'.box
                        |> optionm'.map' fun channel_receiver =>
                            channel_receiver
                            |> threading.arc_mutex_lock'
                            |> resultm.unwrap'
                            |> iter.iter
                            |> iter_collect''
                            |> am'.vec_map sm'.from_std_string
                            |> am'.from_vec
                            |> fun x => x : _ i32 _
                            |> seq.of_array
                            |> sm'.concat "\n"
                        |> optionm'.unbox
                        |> optionm.map (sm'.trim >> sm'.to_option)
                        |> optionm'.flatten
                        |> optionm'.default_value (
                            std_trace
                            |> optionm.map sm'.from_std_string
                            |> optionm'.default_value ""
                        )
                    trace Verbose
                        fun () => "runtime.execute_with_options / result"
                        fun () => { file_name exit_code std_trace_len = ##std_trace }
                    new_pair exit_code std_trace
        |> capture
    |> fun x => x ()
    |> from_pair

### execute_with_options_rust

In [ ]:
inl execute_with_options_rust (options : execution_options) : i32 * string =
    inl command = join options.command
    inl file_name, arguments = command |> split_command |> resultm.get
    inl arguments =
        arguments
        |> optionm'.default_value ""
        |> split_args
        |> resultm.get
        |> am'.to_vec
        |> am'.vec_map sm'.to_std_string
    trace Debug
        fun () => "runtime.execute_with_options"
        fun () => { file_name arguments = #?*arguments }
    inl trace' = options.trace
    inl on_stdout (line : resultm.result' sm'.std_string sm'.std_string) : sm'.std_string =
        line |> stdio_line_rust (Ok ()) trace'
    inl on_stderr (line : resultm.result' sm'.std_string sm'.std_string) : sm'.std_string =
        line |> stdio_line_rust (Error ()) trace'
    inl kind, exit_code, error, lines =
        !\\(
            (
                file_name,
                arguments,
                options.working_directory,
                options.environment_variables,
                options.stdin,
                options.cancellation_token,
                options.stderr,
                on_stdout,
                on_stderr
            ),
            $'"{ let wd: Option<std::rc::Rc<str>> = $2.clone(); let stdin_fn: Option<std::rc::Rc<dyn Fn(std::sync::Arc<std::sync::Mutex<std::process::ChildStdin>>)>> = $4.clone(); let token: Option<std::sync::Arc<std::sync::atomic::AtomicBool>> = $5.clone(); let stderr_on: bool = $6; let mut c = std::process::Command::new(&*$0); c.args(&*$1); c.stdout(std::process::Stdio::piped()); c.stderr(std::process::Stdio::piped()); c.stdin(std::process::Stdio::piped()); if let Some(d) = &wd { c.current_dir(&**d); } for (k, v) in $3.borrow().iter() { c.env(&**k, &**v); } match c.spawn() { Err(e) => (1i32, None, Some(format\!(\\\"{}\\\", e)), Vec::new()), Ok(mut child) => { let out = child.stdout.take().unwrap(); let err = child.stderr.take().unwrap(); let (tx, rx) = std::sync::mpsc::channel::<(bool, Result<std::string::String, std::string::String>)>(); let tx2 = tx.clone(); let h1 = std::thread::spawn(move || { for l in std::io::BufRead::lines(std::io::BufReader::new(encoding_rs_io::DecodeReaderBytesBuilder::new().utf8_passthru(true).build(out))) { if tx.send((false, l.map_err(|e| format\!(\\\"{}\\\", e)))).is_err() { break; } } }); let h2 = std::thread::spawn(move || { for l in std::io::BufRead::lines(std::io::BufReader::new(encoding_rs_io::DecodeReaderBytesBuilder::new().utf8_passthru(true).build(err))) { if stderr_on && tx2.send((true, l.map_err(|e| format\!(\\\"{}\\\", e)))).is_err() { break; } } }); if let (Some(f), Some(s)) = (&stdin_fn, child.stdin.take()) { let a = std::sync::Arc::new(std::sync::Mutex::new(s)); f(a.clone()); let _ = std::io::Write::flush(&mut *a.lock().unwrap()); } let mut lines: Vec<std::string::String> = Vec::new(); let mut cancelled = false; loop { if let Some(t) = &token { if t.load(std::sync::atomic::Ordering::SeqCst) { let _ = child.kill(); cancelled = true; break; } } match rx.recv_timeout(std::time::Duration::from_millis(100)) { Ok((false, l)) => lines.push($7(l)), Ok((true, l)) => lines.push($8(l)), Err(std::sync::mpsc::RecvTimeoutError::Timeout) => {}, Err(std::sync::mpsc::RecvTimeoutError::Disconnected) => break } } if \!cancelled { let _ = h1.join(); let _ = h2.join(); } match child.wait() { Ok(s) => (0i32, s.code(), None, lines), Err(e) => (2i32, None, Some(format\!(\\\"{}\\\", e)), lines) } } } }"'
        ) : i32 * optionm'.option' i32 * optionm'.option' sm'.std_string * am'.vec sm'.std_string
    inl error = error |> optionm'.default_value' (sm'.to_std_string "") |> sm'.from_std_string
    inl exit_code, std_trace =
        if kind = 1 then
            trace Critical
                fun () => "runtime.execute_with_options / child error"
                fun () => { error file_name arguments = #?*arguments }
            -1i32, error
        elif kind = 2 then
            trace Critical
                fun () => "runtime.execute_with_options / output error"
                fun () => { error file_name arguments = #?*arguments }
            -2i32, error
        else
            inl exit_code, fallback =
                match exit_code |> optionm'.unbox with
                | Some exit_code => exit_code, ""
                | None => -1i32, "runtime.execute_with_options / exit_code=None"
            inl text =
                lines
                |> am'.vec_map sm'.from_std_string
                |> am'.from_vec
                |> fun x => x : _ i32 _
                |> seq.of_array
                |> sm'.concat "\n"
                |> sm'.trim
            exit_code, (if text = "" then fallback else text)
    trace Verbose
        fun () => "runtime.execute_with_options / result"
        fun () => { file_name exit_code std_trace_len = ##std_trace }
    exit_code, std_trace

### execute_with_options

In [ ]:
let execute_with_options (options : execution_options) : i32 * string =
    backend_switch {
        Fsharp = fun () => execute_with_options_fsharp options
        Rust = fun () => execute_with_options_rust options
    }

#### execute

In [ ]:
let execute command =
    execution_options fun x => { x with
        command = command
    }
    |> execute_with_options

#### tests

In [6]:
///- --test
///> rust -d chrono encoding_rs encoding_rs_io regex sha2

inl content = "╭─[ 你好，世界！こんにちは世界！ ]─╮"

inl file_name = join "test.txt"
inl temp_dir, disposable =
    #?*(file_name, content)
    |> crypto.hash_text
    |> file_system.create_temp_dir'
disposable |> use |> ignore
inl path = temp_dir </> file_name |> file_system.normalize_path
inl exit_code, result =
    execute ("pwsh -c \"[IO.File]::ReadAllText('" ++# path ++# "')\"")
exit_code |> _assert_eq 1
result |> _assert sm'.contains "not find file"

content |> file_system.write_all_text path

inl exit_code, output =
    execution_options fun x => { x with
        command = "pwsh -c \"[System.Console]::OutputEncoding = [System.Text.Encoding]::UTF8; [IO.File]::ReadAllText('" ++# file_name ++# "')\""
        working_directory = Some temp_dir |> optionm'.box
    }
    |> execute_with_options

exit_code |> _assert_eq 0i32
output |> _assert_eq content

13:01:55 1v file_system.create_dir / { dir = /tmp/!create_temp_path_/spiral_cf21d07b0367630844927a5d87bc9f7d42c89948b85f5fe169ffca165033aee9/664346c7-6234-0aa5-c6c9-79ffbb32476f }
13:01:55 2d runtime.execute_with_options / { file_name = pwsh; arguments = ["-c", "[IO.File]::ReadAllText('/tmp/!create_temp_path_/spiral_cf21d07b0367630844927a5d87bc9f7d42c89948b85f5fe169ffca165033aee9/664346c7-6234-0aa5-c6c9-79ffbb32476f/test.txt')"] }
13:01:55 3v ! MethodInvocationException: Exception calling "ReadAllText" with "1" argument(s): "Could not find file '/tmp/!create_temp_path_/spiral_cf21d07b0367630844927a5d87bc9f7d42c89948b85f5fe169ffca165033aee9/664346c7-6234-0aa5-c6c9-79ffbb32476f/test.txt'."
13:01:55 4v runtime.execute_with_options / result / { file_name = pwsh; exit_code = 1; std_trace_len = 275 }
__assert_eq { expected = 1 }
__assert { expected = MethodInvocationException: Exception calling "ReadAllText" with "1" argument(s): "Could not find file '/tmp/!create_temp_path_/spiral_cf21d07b0

In [7]:
///- --test
///> rust -d chrono encoding_rs encoding_rs_io regex sha2

inl exit_code, output =
    execution_options fun x => { x with
        command = "pwsh -c \"[System.Console]::OutputEncoding = [System.Text.Encoding]::UTF8; for ($i = 0; $i -lt 5; $i++) { Write-Output `\"Line$i`\"; Start-Sleep -Milliseconds 200 }\""
    }
    |> execute_with_options
exit_code |> _assert_eq 0i32
output |> _assert sm'.contains "Line4"

13:02:02 1d runtime.execute_with_options / { file_name = pwsh; arguments = ["-c", "[System.Console]::OutputEncoding = [System.Text.Encoding]::UTF8; for ($i = 0; $i -lt 5; $i++) { Write-Output `\"Line$i`\"; Start-Sleep -Milliseconds 200 }"] }
13:02:02 2v > "Line0"
13:02:02 3v > "Line1"
13:02:02 4v > "Line2"
13:02:02 5v > "Line3"
13:02:03 6v > "Line4"
13:02:03 7v runtime.execute_with_options / result / { file_name = pwsh; exit_code = 0; std_trace_len = 39 }
__assert_eq { expected = 0 }
__assert { expected = "Line0"
"Line1"
"Line2"
"Line3"
"Line4" }

### execute_retry

In [ ]:
let execute_retry retries options =
    fun () =>
        inl exit_code, result = options |> execute_with_options
        if exit_code = 0
        then Ok (exit_code, result)
        else Error (exit_code, result)
    |> retry_fn' retries

## main

In [8]:
///> fsharp

inl main () =
    init_trace_state None
    $'let current_process_kill () = !current_process_kill ()' : ()
    inl execution_options fn =
        execution_options fun x =>
            x
            |> heap
            |> fn
            |> fun x => !x
    $'let execution_options x = !execution_options x' : ()
    inl split_args x = x |> split_args |> resultm.box
    $'let split_args x = !split_args x' : ()

/home/runner/work/spiral/spiral/lib/spiral/main.fsx(2291,23): warning FS0025: Incomplete pattern matches on this expression. For example, the value 'UH0_Cons (_, _)' may indicate a case not covered by the pattern(s).